In [ ]:
import numpy as np
import pandas as pd
import timeit
import pickle
import torch
import torch.nn as nn
import yaml

Floatt = torch.float64

from Ordinal_ResLogit.models import OrdinalResLogit, ResLogit, Logit

In [ ]:
#read data
raw_data = pd.read_csv("Pedestrian_Waittime.csv")

#define the configuration file
with open ("config.yaml") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

#define the inputs and output
x_data = raw_data.iloc[:,5:21]
y_data=raw_data["category"] 

#add the list of explanatory variable names to the configuration file
config["variables"] = list(x_data.columns)

#number of observations, variables and choices 
config["n_obs"] = raw_data.shape[0]
config["n_vars"] = x_data.shape[1]
config["n_choices"] = len(config["choices"])

# slicing index for train and valid split
slice = np.floor(0.7 * config["n_obs"]).astype(int)
config["slice"] = slice

#slices x and y datasets into train and valid datasets based on slice
train_x_data, valid_x_data = x_data.iloc[:slice], x_data.iloc[slice:]
train_y_data, valid_y_data = y_data.iloc[:slice], y_data.iloc[slice:]

#number of train and valid batches 
n_train_batches = train_y_data.shape[0] // config["batch_size"]
n_valid_batches = valid_y_data.shape[0] // config["batch_size"]
config["n_train_batches"] = n_train_batches
config["n_valid_batches"] = n_valid_batches

#define ordinal levels for train and valid dataset
train_level_data = pd.DataFrame(columns = ["level1"])
for i in range (train_y_data.shape[0]):
    label_train = y_data[i]
    classifier_train = [1] * (label_train -1) + [0] *(config["n_choices"] - label_train)
    train_level_data_lenght = len(train_level_data)
    train_level_data.loc[train_level_data_lenght] = classifier_train
    
valid_level_data =pd.DataFrame(columns = ["level1"])
for i in range (valid_y_data.shape[0]):
    label_valid = y_data[i + train_y_data.shape[0]]
    classifier_valid = [1] * (label_valid -1) + [0] *(config["n_choices"] - label_valid)
    valid_level_data_length = len(valid_level_data)
    valid_level_data.loc[valid_level_data_length] = classifier_valid
    

# convert to Pytorch tensor 
train_x_tensor = torch.as_tensor(train_x_data.values , dtype = Floatt)
train_y_tensor = torch.as_tensor(train_y_data.values , dtype = torch.int64)
valid_x_tensor = torch.as_tensor(valid_x_data.values , dtype = Floatt)
valid_y_tensor = torch.as_tensor(valid_y_data.values , dtype = torch.int64)
train_level_tensor = torch.as_tensor(train_level_data.values , dtype = torch.int64)
valid_level_tensor = torch.as_tensor(valid_level_data.values , dtype = torch.int64)

In [ ]:
class Training(object):
    def main_model_OrdinalResLogit(self, x, y):
        self.model = OrdinalResLogit(input = x, 
                                    choice = y, 
                                    n_vars = config["n_vars"], 
                                    n_choices = config["n_choices"],
                                    n_layers = config["n_layers"], 
                                    batch_size=config["batch_size"])
        
        self.cost = nn.BCEWithLogitsLoss(reduction = "sum")
        self.opt = torch.optim.RMSprop(self.model.params,lr=config['learning_rate'], alpha=0.9, eps=1e-10, weight_decay=0)
        
    def main_model_ResLogit(self, x, y):
        self.model = ResLogit(input = x, choice = y, 
                                    n_vars = config["n_vars"], 
                                    n_choices = config["n_choices"],
                                    n_layers = config["n_layers"])
        

        self.cost = nn.CrossEntropyLoss(reduction = "sum")
        self.opt = torch.optim.RMSprop(self.model.params,lr=config['learning_rate'], alpha=0.9, eps=1e-10, weight_decay=0)
        
    def main_model_Logit(self, x, y):
        self.model = Logit(input = x, choice = y, 
                                    n_vars = config["n_vars"], 
                                    n_choices = config["n_choices"])
        
        self.cost = nn.CrossEntropyLoss(reduction = "sum")
        self.opt = torch.optim.RMSprop(self.model.params,lr=config['learning_rate'], alpha=0.9, eps=1e-10, weight_decay=0)
        
    def train_model(self, inputs, choice):
        self.model.fit(inputs)
        loss = self.cost(self.model.output, choice)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        
        return loss
    
    def validate_model(self, valid_x, valid_y):
        with torch.no_grad():
            self.model.fit(valid_x)
            valid_loss = self.cost(self.model.output, valid_y)
            
            return valid_loss
        
        
    def error(self, choice):
        errors = self.model.errors(choice)

        return errors 
    
    
    def predict_validate(self, inputs):
        pred = self.model.predict(inputs)
        
        return pred
    
    
    def prob(self, inputs):
        self.model.fit(inputs)

        return self.model.output_likelihood
    
    def hessian(self, inputs, category ="low"):
        self.model.fit(inputs)
        prob_fun = torch.tensor(self.model.output_likelihood, requires_grad=True)
        if category == "high":
            dpdx = torch.autograd.grad(prob_fun.sum(axis=0), inputs)
        else:
            dpdx = torch.autograd.grad(1-prob_fun.sum(axis=0), inputs)

        return dpdx

In [ ]:
#creat model
training_object = Training()

if config['model_type'] == 'OrdinalResLogit':
    # create ResNet model
    training_object.main_model_OrdinalResLogit(train_x_tensor, train_level_tensor)
    
elif config['model_type'] == 'ResLogit':
    # create ResNet model
    training_object.main_model_ResLogit(train_x_tensor, train_y_tensor)
    
elif config['model_type'] == 'MNL':
    # create ResNet model
    training_object.main_model_Logit(train_x_tensor, train_y_tensor)
    
print(config['model_type'])

In [ ]:
#training loop
epoch = 0
valid_freq = min(200, n_train_batches)
best_validation_ll = np.inf
batch_size = config["batch_size"]
start_time = timeit.default_timer()
done_looping = False
step = 0


filename = '{}{}_bestmodel.pkl'.format(config['model_type'], config['n_layers'])
training_frame = pd.DataFrame(columns=['epoch', 'minibatch', 'batches', 'train_ll', 'valid_ll', 'valid_err'])

while (epoch < config['n_epochs']) and (not done_looping):
    epoch = epoch + 1
    training_ll = 0
    
    # Minibatch loop
    for i in range(n_train_batches):
        inputs = train_x_tensor[i * batch_size : (i+1) * batch_size]
        if config['model_type'] == 'OrdinalResLogit':
            choice = train_level_tensor[i * batch_size : (i+1) * batch_size].double()
        else:
            choice = train_y_tensor[i * batch_size : (i+1) * batch_size]-1
            
        minibatch_ll = training_object.train_model(inputs, choice).item()
        training_ll = (training_ll * i + minibatch_ll)/(i + 1)
        
        iteration = (epoch - 1) * n_train_batches + i
            
        # If the current iter has reached the valid_freq then evaluate the validation loss on the validation batches
        if (iteration + 1) % valid_freq == 0:
            if config['model_type'] == 'OrdinalResLogit':
                validation_ll = training_object.validate_model(valid_x_tensor, valid_level_tensor.double()).item()
            else:
                validation_ll = training_object.validate_model(valid_x_tensor, (valid_y_tensor-1)).item()
            
            
            #check prediction accuracy
            error = training_object.error(valid_y_tensor).item()
            
            #################################
            # track and save training stats #
            #################################
            training_step = {
                'epoch': epoch, 
                'minibatch': i + 1, 
                'batches': n_train_batches, 
                'train_ll': training_ll * n_train_batches, 
                'valid_ll': validation_ll, 
                'valid_err': error,
            }
            training_frame.loc[step] = training_step
            #################################
            
            
            if validation_ll < best_validation_ll:
                print(('epoch {:d}, minibatch {:d}/{:d}, '
                       'validation likelihood {:.2f}').format(
                        epoch, i + 1, n_train_batches, validation_ll))

                # improve patience if loss improvement is good enough
                if validation_ll < best_validation_ll * config['improvement_threshold']:
                    config['patience'] = max(config['patience'], iteration * config['patience_increase'])
                
                
                # set the best loss to the new current (good) validation loss
                best_validation_ll = validation_ll
                
                error = training_object.error(valid_y_tensor).item()
                training_frame.loc[step, 'valid_err'] = error
                print('validation error  {:.2%}'.format(error))
            
                # save the best model
                with open(filename, 'wb') as f:
                    pickle.dump([training_object, config], f)

            step = step + 1
            
        if epoch > 200:
            done_looping = True
            break
      
end_time = timeit.default_timer()
run_time = end_time - start_time
print(run_time)

In [ ]:
#analyze the accuracy of model
y_pred_valid = training_object.predict_validate(valid_x_tensor)

# Count the number alternatives
num_low = torch.sum(y_pred_valid == 1)
num_high = torch.sum(y_pred_valid == 2)

from sklearn import metrics
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import normalize
print('confusion_matrix for validation data:')
print(confusion_matrix(valid_y_tensor,y_pred_valid))
print(classification_report(valid_y_tensor,y_pred_valid))